# EDA

Exploratory Data Analysis of the jurisprudence datasets.

In [ ]:
import json
import pandas as pd
from pathlib import Path
import re

DATA_DIR = Path('../../data/1_raw')

In [ ]:
def field_info(df, n=3):
    rows = []
    for col in df.columns:
        samples = df[col].dropna().head(n).tolist()
        samples = [str(v)[:60] + '...' if isinstance(v, str) and len(v) > 60 else v for v in samples]
        rows.append({'field': col, 'dtype': str(df[col].dtype), 'examples': samples})
    return pd.DataFrame(rows)

## 1. Data Loading

### 1.1 Load the Datasets

#### STJ (Supremo Tribunal da Justiça)

In [ ]:
with open(DATA_DIR / 'STJ.json') as f:
    stj = pd.DataFrame(json.load(f))

print(f'Records : {stj.shape[0]}')
print(f'Fields  : {stj.shape[1]}')
stj.head(2)

In [ ]:
field_info(stj)

#### STA (Supremo Tribunal Administrativo)

In [ ]:
with open(DATA_DIR / 'STA.json') as f:
    sta = pd.DataFrame(json.load(f))

print(f'Records : {sta.shape[0]}')
print(f'Fields  : {sta.shape[1]}')
sta.head(2)

In [ ]:
field_info(sta)

#### TCAN (Tribunal Central Administrativo Norte)

In [ ]:
with open(DATA_DIR / 'TCAN.json') as f:
    tcan = pd.DataFrame(json.load(f))

print(f'Records : {tcan.shape[0]}')
print(f'Fields  : {tcan.shape[1]}')
tcan.head(2)

In [ ]:
field_info(tcan)

#### TCAS (Tribunal Central Administrativo Sul)

In [ ]:
with open(DATA_DIR / 'TCAS.json') as f:
    tcas = pd.DataFrame(json.load(f))

print(f'Records : {tcas.shape[0]}')
print(f'Fields  : {tcas.shape[1]}')
tcas.head(2)

In [ ]:
field_info(tcas)

#### TRE (Tribunal da Relação de Évora)

In [ ]:
with open(DATA_DIR / 'TRE.json') as f:
    tre = pd.DataFrame(json.load(f))

print(f'Records : {tre.shape[0]}')
print(f'Fields  : {tre.shape[1]}')
tre.head(2)

In [ ]:
field_info(tre)

#### TRG (Tribunal da Relação de Guimarães)

In [ ]:
with open(DATA_DIR / 'TRG.json') as f:
    trg = pd.DataFrame(json.load(f))

print(f'Records : {trg.shape[0]}')
print(f'Fields  : {trg.shape[1]}')
trg.head(2)

In [ ]:
field_info(trg)

#### TRL (Tribunal da Relação de Lisboa)

In [ ]:
with open(DATA_DIR / 'TRL.json') as f:
    trl = pd.DataFrame(json.load(f))

print(f'Records : {trl.shape[0]}')
print(f'Fields  : {trl.shape[1]}')
trl.head(2)

In [ ]:
field_info(trl)

#### TRP (Tribunal da Relação do Porto)

In [ ]:
with open(DATA_DIR / 'TRP.json') as f:
    trp = pd.DataFrame(json.load(f))

print(f'Records : {trp.shape[0]}')
print(f'Fields  : {trp.shape[1]}')
trp.head(2)

In [ ]:
field_info(trp)

#### TRC (Tribunal da Relação de Coimbra)

In [ ]:
with open(DATA_DIR / 'TRC.json') as f:
    trc = pd.DataFrame(json.load(f))

print(f'Records : {trc.shape[0]}')
print(f'Fields  : {trc.shape[1]}')
trc.head(2)

In [ ]:
field_info(trc)

#### TConflitos (Tribunal de Conflitos)

In [ ]:
with open(DATA_DIR / 'TConflitos.json') as f:
    tconflitos = pd.DataFrame(json.load(f))

print(f'Records : {tconflitos.shape[0]}')
print(f'Fields  : {tconflitos.shape[1]}')
tconflitos.head(2)

In [ ]:
field_info(tconflitos)

#### Global Table

In [ ]:
summary = pd.DataFrame([
    {'source': 'STJ',        'records': stj.shape[0],        'fields': stj.shape[1]},
    {'source': 'STA',        'records': sta.shape[0],        'fields': sta.shape[1]},
    {'source': 'TCAN',       'records': tcan.shape[0],       'fields': tcan.shape[1]},
    {'source': 'TCAS',       'records': tcas.shape[0],       'fields': tcas.shape[1]},
    {'source': 'TRE',        'records': tre.shape[0],        'fields': tre.shape[1]},
    {'source': 'TRG',        'records': trg.shape[0],        'fields': trg.shape[1]},
    {'source': 'TRL',        'records': trl.shape[0],        'fields': trl.shape[1]},
    {'source': 'TRP',        'records': trp.shape[0],        'fields': trp.shape[1]},
    {'source': 'TRC',        'records': trc.shape[0],        'fields': trc.shape[1]},
    {'source': 'TConflitos', 'records': tconflitos.shape[0], 'fields': tconflitos.shape[1]},
])
summary

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(summary['source'], summary['records'], color='steelblue')
ax.set_xlabel('Source')
ax.set_ylabel('Records')
ax.set_title('Class distribution across sources')
for bar, val in zip(bars, summary['records']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

### 1.2 Target variable and Sumario feature (main)

In [ ]:
all_data = pd.concat([
    stj.assign(source='STJ'),
    sta.assign(source='STA'),
    tcan.assign(source='TCAN'),
    tcas.assign(source='TCAS'),
    tre.assign(source='TRE'),
    trg.assign(source='TRG'),
    trl.assign(source='TRL'),
    trp.assign(source='TRP'),
    trc.assign(source='TRC'),
    tconflitos.assign(source='TConflitos'),
], ignore_index=True)

In [ ]:
def parse_decisao(obs):
    if not obs or not isinstance(obs, str):
        return None
    m = re.search(r'Decis[aã]o:\s*(.+)', obs)
    return m.group(1).strip() if m else None

all_data['Decisao'] = all_data['Observacoes'].apply(parse_decisao)

print(f'Registos com Decisao : {all_data["Decisao"].notna().sum()} / {len(all_data)}')
print(f'Valores únicos       : {all_data["Decisao"].nunique()}')
print()
all_data['Decisao'].value_counts(dropna=False)

#### Variável alvo — `Decisao`

In [ ]:
tipo_counts = all_data['Decisao'].value_counts(dropna=False)
print(f'Unique classes : {all_data["Decisao"].nunique()}')
print(f'Null values    : {all_data["Decisao"].isna().sum()}')
tipo_counts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
tipo_counts.dropna().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Decisao')
ax.set_ylabel('Registos')
ax.set_title('Distribuição da variável alvo (Decisao)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

#### Main feature — `Sumario`

In [ ]:
sumario = all_data['Sumario']
print(f'Null values   : {sumario.isna().sum()}')
print(f'Empty strings : {(sumario == "").sum()}')
print()
sumario.str.len().describe().rename('char_length')

In [ ]:
(
    all_data.groupby('source')['Sumario']
    .agg(
        nulls=lambda s: s.isna().sum(),
        empty=lambda s: (s == '').sum(),
        count='count',
        mean=lambda s: s.str.len().mean(),
        std=lambda s: s.str.len().std(),
        min=lambda s: s.str.len().min(),
        median=lambda s: s.str.len().median(),
        max=lambda s: s.str.len().max(),
    )
    .round(1)
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sumario.str.len().dropna().plot(kind='hist', bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Character length')
ax.set_ylabel('Count')
ax.set_title('Sumario length distribution')
plt.tight_layout()
plt.show()

Checking manually the null values.

In [ ]:
null_idx = all_data[all_data['Sumario'].isna()].index.tolist()
print(f'Null indices ({len(null_idx)} total): {null_idx[:20]} ...')

i = null_idx[10]
all_data.loc[i, ['source', 'OrigemEnum', 'NumeroProcesso', 'Sumario', 'Texto']].to_frame('value')

In [ ]:
out_dir = Path('../../null_values')
out_dir.mkdir(exist_ok=True)

row = all_data.loc[i]

out_path = out_dir / f'idx_{i}.txt'
with open(out_path, 'w', encoding='utf-8') as f:
    for col in all_data.columns:
        f.write(f'=== {col} ===\n{row[col]}\n\n')

print(f'Saved to {out_path.resolve()}')

### 1.3 Taxonomia da Variável Alvo — `Decisao`

In [ ]:
all_data['Decisao'].value_counts().to_frame('registos')

In [ ]:
eligible_mask = (
    all_data['Sumario'].notna()
    & all_data['Sumario'].str.strip().ne('')
    & all_data['Decisao'].notna()
)
unique_decisao = all_data.loc[eligible_mask, 'Decisao'].value_counts()

out_path = Path('../../decisao_raw_values.txt')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(f'# Raw Decisao values from eligible records (Sumario present)\n')
    f.write(f'# Eligible records : {eligible_mask.sum()}\n')
    f.write(f'# Unique values    : {len(unique_decisao)}\n\n')
    for val, cnt in unique_decisao.items():
        f.write(f'{cnt:5d}  {val}\n')

print(f'Saved {len(unique_decisao)} unique Decisao values → {out_path}')

In [ ]:
def categorize_decisao(raw):
    if not raw or not isinstance(raw, str):
        return 'OUTRO'
    v = raw.upper().strip()
    if re.search(r'PARCIAL|EM PARTE', v):
        return 'PARCIAL'
    if re.search(r'IMPROCEDENTE|IMPROCEDÊN|NEGADO|NÃO PROVID|NEGAR|NÃO PROCEDENTE|NEGADA|NÃO ADMITID|NÃO CONHECIMENTO|NÃO ADMISS|DESATENDID|NEGA-SE', v):
        return 'IMPROCEDENTE'
    if re.search(r'CONFIRMAD|CONFIRMAÇ', v):
        return 'CONFIRMADA'
    if re.search(r'REVOGAD|REVOGAÇ', v):
        return 'REVOGADA'
    if re.search(r'INDEFERI|REJEIÇÃO|REJEITAD', v):
        return 'INDEFERIDA'
    if re.search(r'PROCEDENTE|PROCEDÊN|PROVIDO|PROVIDA|CONCEDID|PROVIMENTO|DEFERIMENT|DEFERIDA\b|DEFERIDO\b|ADMITIDA A REVISTA', v):
        return 'PROCEDENTE'
    if re.search(r'ANULAD|ANULAÇ|NULIDADE', v):
        return 'ANULADA'
    if re.search(r'ALTERAD|MODIFICAD', v):
        return 'ALTERADA'
    if re.search(r'RESOLV|DIRIMI', v):
        return 'RESOLVIDO'
    return 'OUTRO'

all_data['Decisao_cat'] = all_data['Decisao'].apply(categorize_decisao)

cat_counts = all_data.loc[eligible_mask, 'Decisao_cat'].value_counts()
print(cat_counts.to_frame('registos').to_string())
print(f'\nTotal eligible : {eligible_mask.sum()}')
print(f'OUTRO          : {cat_counts.get("OUTRO", 0)} ({cat_counts.get("OUTRO", 0) / eligible_mask.sum() * 100:.1f}%)')

## 2. Data Quality Assessment

In [ ]:
total = len(all_data)

missing = pd.DataFrame({
    'null_count':   all_data.isnull().sum(),
    'empty_count':  (all_data == '').sum(),
}).assign(
    total_missing=lambda df: df['null_count'] + df['empty_count'],
    pct=lambda df: (df['total_missing'] / total * 100).round(2),
).sort_values('total_missing', ascending=False)

missing[missing['total_missing'] > 0]

#### Sumario — missing breakdown by source

In [ ]:
sumario_missing = (
    all_data.assign(missing=all_data['Sumario'].isna() | (all_data['Sumario'] == ''))
    .groupby('source')
    .agg(total=('missing', 'count'), missing=('missing', 'sum'))
    .assign(pct=lambda df: (df['missing'] / df['total'] * 100).round(2))
    .sort_values('missing', ascending=False)
)
sumario_missing

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = sumario_missing.index
width = 0.4

bars_total   = ax.bar(x, sumario_missing['total'],   width, label='Total', color='steelblue')
bars_missing = ax.bar(x, sumario_missing['missing'],  width, label='Sumario em falta', color='tomato')

for bar, pct in zip(bars_missing, sumario_missing['pct']):
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                f'{pct}%', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos')
ax.set_title('Sumario em falta por tipo de documento')
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Duplicate Records

Two levels of duplication are checked.

In [ ]:
dup_process = all_data[all_data.duplicated('NumeroProcesso', keep=False)].shape[0]
dup_sumario = all_data[all_data['Sumario'].notna()
                       & all_data.duplicated('Sumario', keep=False)].shape[0]

def normalise(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sumario_norm = all_data['Sumario'].dropna().apply(normalise)
near_dup_mask = sumario_norm.duplicated(keep=False)
dup_near = near_dup_mask.sum()

pd.DataFrame({
    'check': [
        'Same NumeroProcesso',
        'Same Sumario (exact)',
        'Same Sumario (normalised)',
    ],
    'affected_regists': [dup_process, dup_sumario, dup_near],
    'pct': [
        round(dup_process / total * 100, 2),
        round(dup_sumario / total * 100, 2),
        round(dup_near    / total * 100, 2),
    ],
})

In [ ]:
near_dup_rows = all_data.loc[sumario_norm[near_dup_mask].index, ['source', 'NumeroProcesso', 'Sumario']].copy()
near_dup_rows['sumario_norm'] = sumario_norm[near_dup_mask]

print(f'Grupos de quasi-duplicados: {near_dup_rows.groupby("sumario_norm").ngroups}')
near_dup_rows.groupby('source').size().rename('quasi_dup_count').to_frame()

In [ ]:
exact_by_source = (
    all_data[all_data['Sumario'].notna() & all_data.duplicated('Sumario', keep=False)]
    .groupby('source').size().rename('exacto')
)
near_by_source = near_dup_rows.groupby('source').size().rename('normalizado')

dup_by_source = pd.concat([exact_by_source, near_by_source], axis=1).fillna(0).astype(int)
dup_by_source = dup_by_source.reindex(all_data['source'].unique()).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(dup_by_source))
w = 0.35
ax.bar([i - w/2 for i in x], dup_by_source['exacto'],      w, color='steelblue',
       label=f'Duplicado exacto ({dup_by_source["exacto"].sum()} / {total})')
ax.bar([i + w/2 for i in x], dup_by_source['normalizado'], w, color='tomato',
       label=f'Quasi-duplicado ({dup_by_source["normalizado"].sum()} / {total})')
ax.set_xticks(list(x))
ax.set_xticklabels(dup_by_source.index)
ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos afectados')
ax.set_title('Duplicados por tribunal (Sumario)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Entradas Malformadas ou Placeholder no Sumario

In [ ]:
PLACEHOLDER_PATTERNS = [
    (r'^\s*$',                          'só espaços'),
    (r'^\s*[.\-_/\\|]{1,3}\s*$',       'só pontuação'),
    (r'^\s*n\.?/?a\.?\s*$',            'N/A'),
    (r'^\s*na[oã]?\s+h[aá]\s*',   'sem sumário (texto)'),
    (r'^\s*sem\s+sum[aá]rio\s*$',       'sem sumário'),
    (r'^\s*.{1,3}\s*$',                 '≤ 3 caracteres'),
    (r'^\s*.{4,9}\s*$',                 '4–9 caracteres'),
]

s = all_data['Sumario'].dropna()
results = []
for pattern, label in PLACEHOLDER_PATTERNS:
    mask = s.str.match(pattern, case=False)
    results.append({
        'categoria':         label,
        'padrão':            pattern,
        'registos':          mask.sum(),
        'pct':               round(mask.sum() / total * 100, 3),
    })

malformed = pd.DataFrame(results)
malformed

#### Sumarios suspeitos — menos de 20 tokens

In [ ]:
TOKEN_THRESHOLD = 20

token_counts = all_data['Sumario'].dropna().str.split().str.len()
short_flag = token_counts < TOKEN_THRESHOLD

flagged = all_data.loc[short_flag[short_flag].index, ['source', 'OrigemEnum', 'NumeroProcesso', 'Sumario']].copy()
flagged['token_count'] = token_counts[short_flag].values

print(f'Registos com Sumario presente mas < {TOKEN_THRESHOLD} tokens : {len(flagged)} ({len(flagged)/total*100:.2f}%)')
print()
flagged.groupby('source')['token_count'].agg(['count', 'min', 'mean', 'max']).round(1)

In [ ]:
flagged.sort_values('token_count').head(10)[['source', 'NumeroProcesso', 'token_count', 'Sumario']]

In [ ]:
out_dir = Path('../../short_sumario')
out_dir.mkdir(exist_ok=True)

examples = flagged.sort_values('token_count').head(20)

for idx, row in examples.iterrows():
    out_path = out_dir / f'idx_{idx}.txt'
    with open(out_path, 'w', encoding='utf-8') as f:
        for col in all_data.columns:
            f.write(f'=== {col} ===\n{all_data.loc[idx, col]}\n\n')

print(f'Guardados {len(examples)} exemplos em {out_dir.resolve()}')

## 3. Target Variable Analysis

In [ ]:
class_counts = all_data['Decisao'].value_counts()
n_classes = class_counts.shape[0]

print(f'Número de classes (Decisao)     : {n_classes}')
print(f'Total de registos com Decisao   : {all_data["Decisao"].notna().sum()} / {len(all_data)}')
print()
class_counts.to_frame('registos').assign(pct=lambda df: (df['registos'] / all_data["Decisao"].notna().sum() * 100).round(2))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

bars = ax1.bar(class_counts.index, class_counts.values, color='steelblue')
for bar, val in zip(bars, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             str(val), ha='center', va='bottom', fontsize=9)
ax1.set_xlabel('Decisao')
ax1.set_ylabel('Registos')
ax1.set_title('Distribuição absoluta')
ax1.tick_params(axis='x', rotation=30)

n_with_decisao = all_data['Decisao'].notna().sum()
pcts = class_counts / n_with_decisao * 100
bars2 = ax2.bar(pcts.index, pcts.values, color='steelblue')
for bar, val in zip(bars2, pcts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax2.set_xlabel('Decisao')
ax2.set_ylabel('Percentagem (%)')
ax2.set_title('Distribuição relativa (sobre registos com Decisao)')
ax2.tick_params(axis='x', rotation=30)

plt.suptitle('Distribuição da variável alvo — Decisao', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

Cobertura.

## Filter — Keep Only Records with `Decisao`

All subsequent analysis requires a labelled target.

In [ ]:
before = len(all_data)
all_data = all_data[all_data['Decisao'].notna()].reset_index(drop=True)
after = len(all_data)

print(f'Dropped  : {before - after:,} records ({(before - after) / before * 100:.1f}%)')
print(f'Remaining: {after:,} records')
all_data['Decisao'].value_counts()

## 4. Sumario Text Analysis

### 4.1 Length & Token-Count Distributions

Metrics computed on all non-null `Sumario` entries: - `char_len`.

In [ ]:
from transformers import AutoTokenizer

df_len = all_data[all_data['Sumario'].notna()].copy()
df_len['char_len']   = df_len['Sumario'].str.len()
df_len['word_count'] = df_len['Sumario'].str.split().str.len()

tok = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
encodings = tok(
    df_len['Sumario'].tolist(),
    add_special_tokens=True,
    truncation=False,
    padding=False,
    return_attention_mask=False,
)
df_len['token_count'] = [len(ids) for ids in encodings['input_ids']]
df_len['over_512']    = df_len['token_count'] > 512

print(f"Entries analysed : {len(df_len):,}")
print(f"Flagged >512 tok : {df_len['over_512'].sum():,}  ({df_len['over_512'].mean()*100:.1f}%)")
print()
df_len[['char_len', 'word_count', 'token_count']].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cols = [('char_len', 'Character length', None),
        ('word_count', 'Word count', None),
        ('token_count', 'Token count (BERT)', 512)]

for ax, (col, label, vline) in zip(axes, cols):
    ax.hist(df_len[col], bins=60, color='steelblue', edgecolor='white')
    ax.axvline(df_len[col].mean(),   color='tomato', linestyle='--', linewidth=1.2, label='mean')
    ax.axvline(df_len[col].median(), color='orange', linestyle='--', linewidth=1.2, label='median')
    if vline:
        ax.axvline(vline, color='crimson', linestyle=':', linewidth=1.5, label=f'{vline}-token limit')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'Global distribution — {label}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
stats_global = (
    df_len[['char_len', 'word_count', 'token_count']]
    .agg(['mean', 'median', 'min', 'max', 'std'])
    .round(1)
    .rename(index={'mean': 'Mean', 'median': 'Median',
                   'min': 'Min', 'max': 'Max', 'std': 'Std'})
)
print('--- Global ---')
display(stats_global)

has_decisao = df_len[df_len['Decisao'].notna()]
stats_per_class = (
    has_decisao
    .groupby('Decisao')[['char_len', 'word_count', 'token_count']]
    .agg(['mean', 'median', 'min', 'max', 'std'])
    .round(1)
)
print('\n--- Per Decisao class ---')
display(stats_per_class)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

classes   = has_decisao['Decisao'].unique()
col_pairs = [('char_len', 'Character length', None),
             ('word_count', 'Word count', None),
             ('token_count', 'Token count (BERT)', 512)]

for ax, (col, label, hline) in zip(axes, col_pairs):
    data = [has_decisao.loc[has_decisao['Decisao'] == c, col].dropna().values for c in classes]
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='white', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
    if hline:
        ax.axhline(hline, color='crimson', linestyle=':', linewidth=1.5, label=f'{hline}-token limit')
        ax.legend(fontsize=9)
    ax.set_xticks(range(1, len(classes) + 1))
    ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Decisao')

plt.tight_layout()
plt.show()

In [ ]:
flag_global = pd.DataFrame({
    'total': [len(df_len)],
    'over_512': [df_len['over_512'].sum()],
    'pct': [round(df_len['over_512'].mean() * 100, 2)]
}, index=['ALL'])

flag_by_source = (
    df_len.groupby('source')
    .agg(total=('over_512', 'count'), over_512=('over_512', 'sum'))
    .assign(pct=lambda d: (d['over_512'] / d['total'] * 100).round(2))
)

flag_by_class = (
    has_decisao.groupby('Decisao')
    .agg(total=('over_512', 'count'), over_512=('over_512', 'sum'))
    .assign(pct=lambda d: (d['over_512'] / d['total'] * 100).round(2))
)

print('--- Global ---');           display(flag_global)
print('\n--- By source ---');       display(flag_by_source)
print('\n--- By Decisao class ---'); display(flag_by_class)

### 4.2 Vocabulary Analysis

In [ ]:
import spacy
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

nlp = spacy.load('pt_core_news_sm', disable=['parser', 'ner', 'lemmatizer'])
PT_STOPWORDS = nlp.Defaults.stop_words

def tokenize(text):
    tokens = re.sub(r'[^\w\s]', ' ', text.lower()).split()
    return [t for t in tokens if t not in PT_STOPWORDS and len(t) > 2 and not t.isdigit()]

def tokenize_str(text):
    return ' '.join(tokenize(text))

In [ ]:
all_tokens = []
for text in df_len['Sumario']:
    all_tokens.extend(tokenize(text))

token_freq = Counter(all_tokens)

total = len(all_tokens)
cumsum, coverage_vocab = 0, 0
for count in sorted(token_freq.values(), reverse=True):
    cumsum += count
    coverage_vocab += 1
    if cumsum / total >= 0.80:
        break

print(f'Total tokens (stopwords removed) : {total:,}')
print(f'Unique vocabulary size           : {len(token_freq):,}')
print(f'Hapax legomena (appear once)     : {sum(1 for v in token_freq.values() if v == 1):,}')
print(f'Tokens covering 80% of corpus    : {coverage_vocab:,}')

In [ ]:
top_n = 30
terms, counts = zip(*token_freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(terms[::-1], counts[::-1], color='steelblue')
ax.set_xlabel('Frequency')
ax.set_title(f'Top {top_n} tokens — global (stopwords removed)')
plt.tight_layout()
plt.show()

In [ ]:
doc_freq = Counter()
for text in df_len['Sumario']:
    doc_freq.update(set(tokenize(text)))

top_n = 30
df_terms, df_counts = zip(*doc_freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_terms[::-1], df_counts[::-1], color='steelblue')
ax.set_xlabel('Document Frequency (# documents)')
ax.set_title(f'Top {top_n} tokens — global (Document Frequency, stopwords removed)')
plt.tight_layout()
plt.show()

In [ ]:
TOP_TERMS = 15
tfidf_rows = []

for cls in sorted(df_len['Decisao'].unique()):
    docs = (
        df_len.loc[df_len['Decisao'] == cls, 'Sumario']
        .dropna()
        .apply(tokenize_str)
        .tolist()
    )
    if len(docs) < 5:
        continue
    vec = TfidfVectorizer()
    X   = vec.fit_transform(docs)
    mean_scores = X.mean(axis=0).A1
    features    = vec.get_feature_names_out()
    for rank, idx in enumerate(mean_scores.argsort()[::-1][:TOP_TERMS], 1):
        tfidf_rows.append({'Decisao': cls, 'rank': rank,
                           'term': features[idx], 'score': round(mean_scores[idx], 4)})

tfidf_df = pd.DataFrame(tfidf_rows)
tfidf_df.pivot(index='rank', columns='Decisao', values='term')

Se a classe tem zero, então significa que tem diversidade lexical dentro de si.

No caso, a classe OUTRO que tem 21 documnetos em que aparece uma frase pelo menos em 30% dos registos, então é repetitivo.

In [ ]:
corpus = df_len['Sumario'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True).tolist()
labels = df_len['Decisao'].tolist()

vec = CountVectorizer(ngram_range=(3, 5), min_df=0.02, max_features=100_000)
X   = vec.fit_transform(corpus)
features = vec.get_feature_names_out()

class_phrases = {}
for cls in sorted(set(labels)):
    idx     = [i for i, l in enumerate(labels) if l == cls]
    X_cls   = X[idx]
    doc_freq = (X_cls > 0).mean(axis=0).A1
    class_phrases[cls] = set(features[doc_freq >= 0.30])

boilerplate = sorted(set.intersection(*class_phrases.values()))
print(f'Boilerplate phrases (≥30% of docs in every class): {len(boilerplate)}')
print()

coverage = {cls: f'{len(class_phrases[cls]):,}' for cls in sorted(class_phrases)}
print('Phrases meeting 30% threshold per class:')
for cls, n in coverage.items():
    print(f'  {cls:<30} {n}')
print()
pd.Series(boilerplate, name='boilerplate_phrase').to_frame().head(30)

### 4.3 Linguistic Patterns

Checks language consistency, detects mixed-language or corrupted entries, and measures structural complexity of the `Sumario` field.

#### Language Detection

In [ ]:
from langdetect import detect, LangDetectException

def detect_lang(text):
    try:
        return detect(text)
    except LangDetectException:
        return 'error'

df_lang = df_len[['Sumario', 'source', 'Decisao']].copy()
df_lang['lang'] = df_lang['Sumario'].apply(detect_lang)

lang_counts = df_lang['lang'].value_counts()
print('Language distribution:')
print(lang_counts.to_string())
print(f'\nPortuguese (pt): {lang_counts.get("pt", 0) / len(df_lang) * 100:.1f}% of entries')


#### Mixed-Language and Corrupted Entries

In [ ]:
non_pt = df_lang[df_lang['lang'] != 'pt'].copy()
print(f'Non-Portuguese entries: {len(non_pt)} ({len(non_pt)/len(df_lang)*100:.1f}%)')
print()

df_lang['alpha_ratio'] = df_lang['Sumario'].str.replace(r'\s', '', regex=True).apply(
    lambda s: sum(c.isalpha() for c in s) / max(len(s), 1)
)
corrupted = df_lang[df_lang['alpha_ratio'] < 0.7]
print(f'Potentially corrupted (alpha ratio < 0.70): {len(corrupted)} entries')
print()

if len(non_pt) > 0:
    display(non_pt[['source', 'Decisao', 'lang', 'Sumario']].head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lang_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Detected language distribution')
axes[0].set_xlabel('Language code')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

axes[1].hist(df_lang['alpha_ratio'], bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(0.7, color='tomato', linestyle='--', label='0.70 threshold')
axes[1].set_title('Alphabetic character ratio')
axes[1].set_xlabel('Ratio')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

#### Sentence Length and Structural Complexity

In [ ]:
nlp_sent = spacy.load('pt_core_news_sm', disable=['ner', 'lemmatizer'])
if 'sentencizer' not in nlp_sent.pipe_names and 'parser' not in nlp_sent.pipe_names:
    nlp_sent.add_pipe('sentencizer')

df_len = df_len.drop(columns=[c for c in ['n_sentences', 'avg_sent_len', 'max_sent_len', 'std_sent_len'] if c in df_len.columns])

all_sent_lens = []
doc_n_sents   = []
doc_std_lens  = []

for text in df_len['Sumario']:
    doc  = nlp_sent(text[:10_000])
    lens = [len(s.text.split()) for s in doc.sents]
    all_sent_lens.extend(lens)
    doc_n_sents.append(len(lens))
    doc_std_lens.append(pd.Series(lens).std() if len(lens) > 1 else 0)

df_len['n_sentences']  = doc_n_sents
df_len['std_sent_len'] = doc_std_lens

sent_lens = pd.Series(all_sent_lens, name='words_per_sentence')

print('--- Sentence length distribution (all sentences flattened) ---')
display(sent_lens.describe().round(1).rename({
    'count': 'total sentences', 'mean': 'mean', '50%': 'median',
    '25%': 'p25', '75%': 'p75'
}))
print(f'\n% sentences > 100 words : {(sent_lens > 100).mean()*100:.1f}%')
print(f'% sentences > 200 words : {(sent_lens > 200).mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(sent_lens.clip(upper=300).to_numpy(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Sentence length (words) — clipped at 300')
axes[0].set_xlabel('Words per sentence')
axes[0].set_ylabel('Count')

axes[1].hist(df_len['n_sentences'].dropna().to_numpy(), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Sentences per document')
axes[1].set_xlabel('# sentences')
axes[1].set_ylabel('Count')

axes[2].hist(df_len['std_sent_len'].dropna().to_numpy(), bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('Std dev sentence length (within doc)')
axes[2].set_xlabel('Std dev (words)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
complexity = (
    df_len.groupby('Decisao')[['n_sentences', 'std_sent_len']]
    .agg(['mean', 'median'])
    .round(2)
    .sort_values(('n_sentences', 'mean'), ascending=False)
)
print('--- Document structure by class ---')
display(complexity)


## 5. Target Taxonomy: Mapping 963 Raw Values to Legal-Effect Classes

The standard approach in legal judgment prediction.

### 5.1 Legal-Effect Classes

### 5.2 Normalization Pipeline & Rule Cascade

In [ ]:
import unicodedata

_TYPO_FIX = [
    (r'PROVIEMNTO|PROVIMRNTO|PROVIMNTO|PRVIMENTO', 'PROVIMENTO'),
    (r'CONFIRNADA|CONFIRMDA|CONFIMACAO|CONFIRNACAO', 'CONFIRMADA'),
    (r'IMPROEDENTE|IMPR-CEDENTE', 'IMPROCEDENTE'),
    (r'INDEFEERIDA|INDEFEDIDA', 'INDEFERIDA'),
    (r'NAGAD', 'NEGAD'),
    (r'PROCDENTE', 'PROCEDENTE'),
    (r'PARCIALMETE\b|PARCIALMEMTE', 'PARCIALMENTE'),
    (r'CENCEDIDA|CONDEDIDA|CONCEDDO', 'CONCEDIDA'),
    (r'RESVISTA|RESVITA', 'REVISTA'),
    (r'RECLAMCAO|RECLAMCAO|RECLAMCAO', 'RECLAMACAO'),
]

def _strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def _normalize(raw):
    if not raw or not isinstance(raw, str):
        return ''
    v = raw.upper().strip()
    v = re.sub(r'[.;,]+$', '', v).strip()
    v = re.sub(r'\s+', ' ', v)
    v = _strip_accents(v)
    for pattern, replacement in _TYPO_FIX:
        v = re.sub(pattern, replacement, v)
    return v

_RULES = [
    ('PARCIALMENTE_PROCEDENTE',
     re.compile(r'PARCIAL|EM PARTE')),

    ('ANULADA_BAIXA',
     re.compile(r'ANULAG?[AO]|ANULAC[AO]O|ANULADO|ANULADA|ANULACAO'
                r'|BAIXA|REMESSA|\bREENVIO\b|AMPLIACAO DA MATERIA'
                r'|NOVO JULGAMENTO|PROCESSADO ANULADO|ANULACAO DA SENTENCA'
                r'|ANULACAO DO ACORDAO')),

    ('NAO_CONHECIDO',
     re.compile(r'NAO CONHECIMENTO|NAO CONHECIDO|\bNAO ADMITID|NAO ADMISS\b'
                r'|INADMISS|\bREJEITAD|\bREJEICAO\b|INUTILIDADE'
                r'|\bEXTINTO\b|NAO SE TOMA CONHECIMENTO|NAO TOMAR CONHECIMENTO'
                r'|NAO RECONHECIMENTO')),

    ('IMPROCEDENTE',
     re.compile(r'IMPROCEDENTE|IMPROCEDENCIA|NEGADO PROVIMENTO|NAO PROVIDO'
                r'|NAO PROVIDA|NAO PROVIDOS|NEGAR PROVIMENTO|\bNEGADA\b'
                r'|\bNEGADO\b|CONFIRMAD|NAO PROCEDENTE|DESATENDID'
                r'|NEGA-SE|\bINDEFERID|INCIDENTE IMPROCEDENTE'
                r'|REVISTA IMPROCEDENTE|RECURSO IMPROCEDENTE')),

    ('PROCEDENTE',
     re.compile(r'\bPROCEDENTE\b|PROCEDENCIA|\bPROVIDO\b|\bPROVIDA\b'
                r'|\bCONCEDIDA\b|\bCONCEDIDO\b|\bDEFERIDA\b|\bDEFERIDO\b'
                r'|\bREVOGAD|\bALTERADA\b|\bALTERADO\b|CONCEDE-SE'
                r'|RECURSO PROVIDO|RECURSO PROCEDENTE|ADMITIDA A REVISTA'
                r'|DEFERIMENTO\b|\bREFORMAD')),
]

def map_legal_effect(raw):
    v = _normalize(raw)
    if not v:
        return 'OUTRO'
    for label, rx in _RULES:
        if rx.search(v):
            return label
    return 'OUTRO'

all_data['Decisao_le'] = all_data['Decisao'].apply(map_legal_effect)
print('Column Decisao_le created.')
print(all_data['Decisao_le'].value_counts())

### 5.3 Rule Cascade Waterfall

For each rule in priority order, report how many records it claims after all higher-priority rules have already fired.

In [ ]:
remaining = all_data['Decisao'].copy()
waterfall = []
classified_idx = pd.Index([])

for label, rx in _RULES:
    norm_remaining = remaining.drop(classified_idx).apply(_normalize)
    hits = norm_remaining[norm_remaining.str.contains(rx.pattern, regex=True, na=False)]
    n = len(hits)
    pct = n / len(all_data) * 100
    waterfall.append({'rule': label, 'records_claimed': n, 'pct_of_total': round(pct, 2)})
    classified_idx = classified_idx.append(hits.index)

n_outro = len(all_data) - len(classified_idx)
waterfall.append({'rule': 'OUTRO', 'records_claimed': n_outro,
                  'pct_of_total': round(n_outro / len(all_data) * 100, 2)})

wf_df = pd.DataFrame(waterfall)
wf_df['cumulative_pct'] = wf_df['pct_of_total'].cumsum().round(2)
display(wf_df)

coverage_5 = wf_df[wf_df['rule'] != 'OUTRO']['pct_of_total'].sum()
print(f'\nCoverage by 5 substantive classes: {coverage_5:.1f}%')
print(f'OUTRO (residual / excluded)       : {100 - coverage_5:.1f}%')

### 5.4 Coverage Audit & OUTRO Residue

OUTRO is reported, not silently dropped.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
le_counts = all_data['Decisao_le'].value_counts()
order = ['PROCEDENTE', 'PARCIALMENTE_PROCEDENTE', 'IMPROCEDENTE',
         'NAO_CONHECIDO', 'ANULADA_BAIXA', 'OUTRO']
vals = [le_counts.get(c, 0) for c in order]
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6', '#3498db', '#95a5a6']
bars = ax.bar(order, vals, color=colors)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(v), ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Records')
ax.set_title('Distribution by legal-effect class')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

total = len(all_data)
for c, v in zip(order, vals):
    print(f'{c:<30} {v:>5}  ({v / total * 100:5.1f}%)')


In [ ]:
outro_records = all_data[all_data['Decisao_le'] == 'OUTRO']['Decisao']
outro_vc = outro_records.value_counts()
print(f'OUTRO: {len(outro_vc)} unique raw values, {outro_records.shape[0]} records')
print()
print(outro_vc.to_string())

### 5.5 Mixed-Party Compound Outcomes

Values containing `/` or `;` encode two parties' appeals with opposite outcomes.

In [ ]:
compound_mask = all_data['Decisao'].str.contains(r'[;/]', regex=True, na=False)
compound_df = (all_data[compound_mask]
               .groupby(['Decisao', 'Decisao_le'])
               .size()
               .reset_index(name='count')
               .sort_values('count', ascending=False))

print(f'Records with compound Decisao: {compound_mask.sum()} '
      f'({compound_mask.sum() / len(all_data) * 100:.1f}% of labelled data)')
print()
print('Class distribution for compound records:')
print(all_data.loc[compound_mask, 'Decisao_le'].value_counts())
print()
print('Top-30 compound values with assigned class:')
display(compound_df.head(30))

### 5.6 Methodological Warnings

Two known pitfalls must be addressed before modelling.

#### 5.6.1 Class Imbalance

`IMPROCEDENTE` will dominate (denial is the modal appellate outcome in virtually every jurisdiction).

In [ ]:
le_dist = all_data['Decisao_le'].value_counts()
total_le = le_dist.sum()

print('Class counts and imbalance ratios:')
print(le_dist.to_frame('count')
      .assign(pct=lambda d: (d['count'] / total_le * 100).round(1),
              imbalance_ratio=lambda d: (le_dist.max() / d['count']).round(1)))

dominant = le_dist.idxmax()
dummy_acc = le_dist.max() / total_le
print(f'\nDominant class : {dominant} ({le_dist.max()} records, {dummy_acc * 100:.1f}% of data)')
print(f'Majority-class dummy accuracy : {dummy_acc * 100:.1f}%')
print('=> Accuracy is uninformative. Use macro-F1 as primary metric.')


#### 5.6.2 Label Leakage in Sumário

If the `Sumario` text explicitly states the outcome (e.g., *'Recurso improcedente'*, *'Negado provimento'*).

In [ ]:
LEAKAGE_PATTERNS = [
    r'recurso improcedente',
    r'negado provimento',
    r'negar provimento',
    r'nao provido',
    r'apelacao improcedente',
    r'revista improcedente',
    r'recurso provido',
    r'recurso procedente',
    r'provido o recurso',
    r'concedida a revista',
    r'negada a revista',
    r'confirmada a sentenca',
    r'confirmada a decisao',
    r'revogada a sentenca',
    r'revogada a decisao',
    r'anulado o acordao',
    r'parcialmente procedente',
    r'parcialmente provido',
]

combined_rx = re.compile('|'.join(LEAKAGE_PATTERNS), re.IGNORECASE)

sumario_clean = all_data['Sumario'].fillna('')
sumario_norm = sumario_clean.apply(lambda s: _strip_accents(s.lower()))

leakage_mask = sumario_norm.str.contains(combined_rx.pattern, regex=True, na=False)
n_leakage = leakage_mask.sum()
print(f'Records where Sumario contains an outcome-revealing phrase: '
      f'{n_leakage} / {len(all_data)} ({n_leakage / len(all_data) * 100:.1f}%)')
print()

rows = []
for pat in LEAKAGE_PATTERNS:
    rx = re.compile(pat, re.IGNORECASE)
    n = sumario_norm.str.contains(rx.pattern, regex=True, na=False).sum()
    rows.append({'pattern': pat, 'n_records': n, 'pct': round(n / len(all_data) * 100, 1)})
display(pd.DataFrame(rows).sort_values('n_records', ascending=False).reset_index(drop=True))

In [ ]:
leak_df = all_data[leakage_mask][['Decisao', 'Decisao_le', 'Sumario']].copy()

sample = leak_df.sample(min(10, len(leak_df)), random_state=42)
for _, row in sample.iterrows():
    sumario_excerpt = row['Sumario'][:200].replace('\n', ' ')
    print(f'Label : {row["Decisao_le"]}')
    print(f'Raw   : {row["Decisao"]}')
    print(f'Text  : {sumario_excerpt}...')
    print()